In [2]:
# import os
# import ssl
# import certifi
# import urllib3

# # Configure SSL certificates
# cert_path = certifi.where()
# os.environ['SSL_CERT_FILE'] = cert_path
# os.environ['REQUESTS_CA_BUNDLE'] = cert_path
# os.environ['AWS_CA_BUNDLE'] = cert_path
# os.environ['CURL_CA_BUNDLE'] = cert_path

# # Create SSL context with proper certificates
# ssl_context = ssl.create_default_context(cafile=cert_path)
# ssl._create_default_https_context = lambda: ssl_context

# # Disable SSL warnings (optional)
# urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# print(f"✅ SSL certificates configured using: {cert_path}")
# print("✅ Environment variables set for AWS, requests, and curl")
# print("✅ Ready to make secure HTTPS connections!")

In [3]:
%reload_ext dotenv
%dotenv

import os
from pathlib import Path

from graphrag_toolkit.lexical_graph import LexicalGraphIndex, set_logging_config
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.load import FileBasedDocs
from graphrag_toolkit.lexical_graph.indexing.build import Checkpoint

# from llama_index.readers.web import SimpleWebPageReader
from llama_index.readers.file import PDFReader

set_logging_config('INFO')

## Extract

In [4]:
extracted_docs = FileBasedDocs(
    docs_directory='extracted'
)

checkpoint = Checkpoint('extraction-checkpoint')

graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

graph_index = LexicalGraphIndex(
    graph_store, 
    vector_store
)

# doc_urls = [
#     'https://docs.aws.amazon.com/neptune/latest/userguide/intro.html',
#     'https://docs.aws.amazon.com/neptune-analytics/latest/userguide/what-is-neptune-analytics.html',
#     'https://docs.aws.amazon.com/neptune-analytics/latest/userguide/neptune-analytics-features.html',
#     'https://docs.aws.amazon.com/neptune-analytics/latest/userguide/neptune-analytics-vs-neptune-database.html'
# ]

# docs = SimpleWebPageReader(
#     html_to_text=True,
#     metadata_fn=lambda url:{'url': url}
# ).load_data(doc_urls)

# graph_index.extract(docs, handler=extracted_docs, checkpoint=checkpoint, show_progress=True)

# ---------------------------------------------------------------------------
#  PDF ingest and extraction
# ---------------------------------------------------------------------------
pdf_dir = Path("data/pdfs")
pdf_dir.mkdir(parents=True, exist_ok=True)

pdf_files = list(pdf_dir.glob("*.pdf"))
if not pdf_files:
    print(f"No PDF files found in {pdf_dir}. "
          "Add PDFs to data/pdfs/ and re-run this cell.")
else:
    print(f"Found {len(pdf_files)} PDF file(s):")
    for pdf in pdf_files:
        print(f"  • {pdf.name}")

    print("\nLoading and extracting documents …")
    # Process each PDF file individually and combine the results
    docs = []
    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}:")
        file_docs = PDFReader().load_data(pdf_file)
        print(f"- Extracted {len(file_docs)} pages")
        
        # Add source metadata to each page
        for i, doc in enumerate(file_docs):
            doc.metadata.update({
                "source": str(pdf_file),
                "page_number": i + 1,
                "total_pages": len(file_docs)
            })
            docs.append(doc)
        
        print(f"- Added metadata to {len(file_docs)} pages")

    print(f"\nTotal documents to process: {len(docs)}")
    
    graph_index.extract(
        docs,
        handler=extracted_docs,
        checkpoint=checkpoint,
        show_progress=True,
    )




collection_id = extracted_docs.collection_id

print('Extraction complete')
print(f'collection_id: {collection_id}')

Found 1 PDF file(s):
  • sample_Pdf.pdf

Loading and extracting documents …

Processing sample_Pdf.pdf:
- Extracted 2 pages
- Added metadata to 2 pages

Total documents to process: 2
2025-07-07 16:32:00:INFO:g.l.i.e.extraction_pipeline:Running extraction pipeline [batch_size: 4, num_workers: 2]


Extracting propositions [nodes: 7, num_workers: 4]: 100%|██████████| 7/7 [00:04<00:00,  1.48it/s]
Extracting propositions [nodes: 9, num_workers: 4]: 100%|██████████| 9/9 [00:06<00:00,  1.45it/s]
Extracting topics [nodes: 9, num_workers: 4]: 100%|██████████| 9/9 [00:21<00:00,  2.40s/it]


2025-07-07 16:32:31:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch_size: 4, num_workers: 1, job_sizes: [441], batch_writes_enabled: True, batch_write_size: 25]
Extraction complete
collection_id: 20250707-163156


## Build

In [5]:
%reload_ext dotenv
%dotenv

import os

from graphrag_toolkit.lexical_graph import LexicalGraphIndex, set_logging_config
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.load import FileBasedDocs
from graphrag_toolkit.lexical_graph.indexing.build import Checkpoint

set_logging_config('INFO')

docs = FileBasedDocs(
    docs_directory='extracted',
    collection_id=collection_id
)
checkpoint = Checkpoint('build-checkpoint')

graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

graph_index = LexicalGraphIndex(
    graph_store, 
    vector_store
)

graph_index.build(docs, checkpoint=checkpoint, show_progress=True)

print('Build complete')

2025-07-07 18:35:32:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch_size: 4, num_workers: 2, job_sizes: [247, 194], batch_writes_enabled: True, batch_write_size: 25]


Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 194/194 [00:00<00:00, 62342.55it/s]
Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 247/247 [00:00<00:00, 53731.29it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 194/194 [00:00<00:00, 410128.52it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 247/247 [00:00<00:00, 370924.84it/s]


Build complete
